# `AAWindowSampler.sample_motif_matched()`

Scan candidate proteins (rows of `df_seq` with no labeled positions) for windows matching a user-supplied PWM, and return the top-scoring hits. Useful for **hard-negative mining**: unlabeled windows that look biochemically similar to the positives at the local-motif level.

In [1]:
import aaanalysis as aa
import pandas as pd
import numpy as np
aa.options["verbose"] = False

df_seq = pd.DataFrame({
    "entry":    ["P1", "P2", "P3"],
    "sequence": ["ACDEFGHIK",
                 "AAACDEFGHIKLMNPQRSTV",
                 "GGGGAAAGGGGAAAGGGGAA"],
    "pos":      [[5], [], []],
})
sampler = aa.AAWindowSampler(random_state=0)

aa_cols = list("ACDEFGHIKLMNPQRSTVWY")
pwm = pd.DataFrame(0.0, index=range(3), columns=aa_cols)
pwm["A"] = 1.0  # prefers A at every position of a length-3 window
aa.display_df(df=pwm, show_shape=True)

DataFrame shape: (3, 20)


,A,C,D,E,F,G,H,I,K,L,M,N,P,Q,R,S,T,V,W,Y
1,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


First call — anchor schema. The `segments` schema is augmented with a `motif_score` column carrying the raw PWM score; results are ranked by descending score and capped at `n`.

In [2]:
df = sampler.sample_motif_matched(df_seq=df_seq, pos_col="pos",
                                  n=5, window_size=3,
                                  motif_pwm=pwm, motif_score_threshold=2.5,
                                  seed=0)
aa.display_df(df=df, show_shape=True)

DataFrame shape: (3, 9)


,entry_win,entry,sequence,window,source_position,label,role,strategy,motif_score
1,P2_1-3,P2,AAACDEFGHIKLMNPQRSTV,AAA,2,0,Negative,motif_matched,3.000000
2,P3_5-7,P3,GGGGAAAGGGGAAAGGGGAA,AAA,6,0,Negative,motif_matched,3.000000
3,P3_12-14,P3,GGGGAAAGGGGAAAGGGGAA,AAA,13,0,Negative,motif_matched,3.000000


## `df_seq`, `pos_col`

Same dual role as `sample_different_protein`: rows with positives are excluded from the scan (and feed the anti-leakage filter), rows with empty `pos_col` cells form the candidate pool. Every position with a fully-fitting window in the candidate pool is scored against `motif_pwm`.

## `n`, `window_size`

`n` caps the number of motif-matched windows returned (after threshold and ranking). `window_size` must equal the first dimension of `motif_pwm`:

In [3]:
df = sampler.sample_motif_matched(df_seq=df_seq, pos_col="pos",
                                  n=2, window_size=3,
                                  motif_pwm=pwm, motif_score_threshold=2.0,
                                  seed=0)
aa.display_df(df=df[["entry_win", "window", "motif_score"]], show_shape=True)

DataFrame shape: (2, 3)


,entry_win,window,motif_score
1,P2_1-3,AAA,3.000000
2,P3_5-7,AAA,3.000000


## `motif_pwm`, `motif_score_threshold`

PWM input shape:

- `pd.DataFrame` (**preferred — safer**) — columns are the 20 canonical amino acids in any order; the validator checks names and reindexes internally to `ACDEFGHIKLMNPQRSTVWY`.
- `np.ndarray` of shape `(window_size, 20)` — columns *must* already be in alphabetical order. The validator cannot detect a wrong order and will silently produce incorrect scores.

`motif_score_threshold` is the score cutoff (sum of per-position values). Both are **required** for this method; non-canonical residues in candidate windows contribute zero:

In [4]:
# Equivalent ndarray PWM (columns implicitly alphabetical).
pwm_arr = pwm.reindex(columns=aa_cols).to_numpy()
df = sampler.sample_motif_matched(df_seq=df_seq, pos_col="pos",
                                  n=3, window_size=3,
                                  motif_pwm=pwm_arr, motif_score_threshold=2.5,
                                  seed=0)
aa.display_df(df=df[["entry_win", "window", "motif_score"]], show_shape=True)

DataFrame shape: (3, 3)


,entry_win,window,motif_score
1,P2_1-3,AAA,3.000000
2,P3_5-7,AAA,3.000000
3,P3_12-14,AAA,3.000000


## `output_mode`

Two output schemas:

- `'segments'` (default) — one row per sampled window, with an extra `motif_score` column.
- `'sequences'` — one row per source protein with a per-residue `labels` list.

In [5]:
df = sampler.sample_motif_matched(df_seq=df_seq, pos_col="pos",
                                  n=3, window_size=3,
                                  motif_pwm=pwm, motif_score_threshold=2.5,
                                  output_mode="sequences", seed=0)
aa.display_df(df=df, show_shape=True)

DataFrame shape: (3, 3)


,entry,sequence,labels
1,P1,ACDEFGHIK,"[None, None, None, None, 1, None, None, None, None]"
2,P2,AAACDEFGHIKLMNPQRSTV,"[None, 0, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]"
3,P3,GGGGAAAGGGGAAAGGGGAA,"[None, None, None, None, None, 0, None, None, None, None, None, None, 0, None, None, None, None, None, None, None]"


## `role`, `label_test`, `label_ref`

Semantic tags. Defaults match hard-negative mining: `role='Negative'`, `label_test=1`, `label_ref=0`:

In [6]:
df = sampler.sample_motif_matched(df_seq=df_seq, pos_col="pos",
                                  n=3, window_size=3,
                                  motif_pwm=pwm, motif_score_threshold=2.5,
                                  role="hard_negative", seed=0)
aa.display_df(df=df[["entry_win", "role", "label", "motif_score"]], show_shape=True)

DataFrame shape: (3, 4)


,entry_win,role,label,motif_score
1,P2_1-3,hard_negative,0,3.000000
2,P3_5-7,hard_negative,0,3.000000
3,P3_12-14,hard_negative,0,3.000000


## `context_cols`, `aa_context_col`, `context_in`, `context_out`

Same provenance and per-residue filtering semantics as the other methods. Providing `context_in` / `context_out` without `aa_context_col` raises:

In [7]:
df_seq_topo = df_seq.assign(topo=["MMMMMMMMM",
                                   "TTTTTTTTTTTTTTTTTTTT",
                                   "TTTTTTTTTTMMMMMMMMMM"])
df = sampler.sample_motif_matched(df_seq=df_seq_topo, pos_col="pos",
                                  n=3, window_size=3,
                                  motif_pwm=pwm, motif_score_threshold=2.5,
                                  aa_context_col="topo", context_in="T",
                                  context_cols=["topo"], seed=0)
aa.display_df(df=df[["entry_win", "window", "motif_score", "topo"]], show_shape=True)

DataFrame shape: (2, 4)


,entry_win,window,motif_score,topo
1,P2_1-3,AAA,3.000000,TTTTTTTTTTTTTTTTTTTT
2,P3_5-7,AAA,3.000000,TTTTTTTTTTMMMMMMMMMM


## `seed`

Per-call seed; falls back to the class-level `random_state`. See `aws_sample_same_protein` for the class-level anti-leakage / redundancy filters (`max_similarity_to_test`, `max_similarity_within_ref`), which apply identically here.

Note the asymmetry with the other sampling methods: `sample_motif_matched` always returns *high-scoring* matches and does not accept `motif_match`. For the inverse (windows that explicitly do **not** match a motif), call `sample_different_protein` with the same `motif_pwm` and `motif_match='out'`.

In [8]:
df_a = sampler.sample_motif_matched(df_seq=df_seq, pos_col="pos",
                                    n=3, window_size=3,
                                    motif_pwm=pwm, motif_score_threshold=2.5,
                                    seed=21)
df_b = sampler.sample_motif_matched(df_seq=df_seq, pos_col="pos",
                                    n=3, window_size=3,
                                    motif_pwm=pwm, motif_score_threshold=2.5,
                                    seed=21)
print("deterministic:", list(df_a["entry_win"]) == list(df_b["entry_win"]))

deterministic: True
